In [1]:
# ============================================================
# PS S6E5 - exp_20260505_013_cat_shared004_no_isoriginaldata_min
# CatBoost shared004 full-FE OOF
#
# Base:
#   012: exp_20260505_012_cat_shared004_full_fe_oof_min
#
# Change from 012:
#   Drop IsOriginalData only
#
# Do not change:
#   - CV split
#   - CatBoost params
#   - original rows concat policy
#   - shared004 full-FE blocks
#   - OOF/pred save format
#   - blend evaluation policy
# ============================================================

In [2]:
import os
import gc
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, log_loss

from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 300)

In [3]:
# ============================================================
# Config
# ============================================================

class CFG:
    EXP_ID = "exp_20260505_013_cat_shared004_no_isoriginaldata_min"
    COMPETITION = "PS S6E5 Predicting F1 Pit Stops"
    METRIC = "AUC"

    TARGET_CANDIDATES = ["PitNextLap", "PitNextLab"]
    TARGET = "PitNextLap"
    ID_COL = "id"
    DANGER_COL = "Normalized_TyreLife"
    
    DROP_RISK_FEATURES = [
        "IsOriginalData",
    ]
    
    COMP_PATHS = [
        "/kaggle/input/competitions/playground-series-s6e5",
        "/kaggle/input/playground-series-s6e5",
        ".",
    ]

    ORIGINAL_PATHS = [
        "/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv",
        "/kaggle/input/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv",
        "./f1_strategy_dataset_v4.csv",
    ]

    OUTDIR = Path(f"/kaggle/working/{EXP_ID}")

    SEED = 42
    N_SPLITS = 5

    USE_ORIGINAL = True

    # CatBoost
    TASK_TYPE = "GPU"
    DEVICES = "0:1"

    ITERATIONS = 9000
    LEARNING_RATE = 0.033
    DEPTH = 8
    L2_LEAF_REG = 7.0
    RANDOM_STRENGTH = 0.8
    EARLY_STOPPING_ROUNDS = 350

    # GPU hidden-default guard
    # SlideShare note:
    #   GPU default border_count / bootstrap_type can differ from CPU.
    #   Fix these explicitly.
    BOOTSTRAP_TYPE = "Bernoulli"
    SUBSAMPLE = 0.8
    BORDER_COUNT = 254

    AUTO_CLASS_WEIGHTS = "Balanced"

    CLIP_LOW = 1e-6
    CLIP_HIGH = 1.0 - 1e-6

    VERBOSE = 200


CFG.OUTDIR.mkdir(parents=True, exist_ok=True)

In [4]:
# ============================================================
# Utilities
# ============================================================

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def print_section(title: str) -> None:
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)


def first_existing_dir(paths):
    for p in paths:
        path = Path(p)
        if (path / "train.csv").exists() and (path / "test.csv").exists():
            return path
    raise FileNotFoundError(f"No valid competition path found: {paths}")


def first_existing_file(paths, required=True):
    for p in paths:
        path = Path(p)
        if path.exists():
            return path
    if required:
        raise FileNotFoundError(f"No valid file path found: {paths}")
    return None


def resolve_target_column(df: pd.DataFrame) -> str:
    for c in CFG.TARGET_CANDIDATES:
        if c in df.columns:
            return c
    raise ValueError(f"No target column found among {CFG.TARGET_CANDIDATES}")


def reduce_mem_usage(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    for col in out.columns:
        dtype = out[col].dtype

        if pd.api.types.is_integer_dtype(dtype):
            c_min = out[col].min()
            c_max = out[col].max()

            if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                out[col] = out[col].astype(np.int8)
            elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                out[col] = out[col].astype(np.int16)
            elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                out[col] = out[col].astype(np.int32)

        elif pd.api.types.is_float_dtype(dtype):
            out[col] = pd.to_numeric(out[col], downcast="float")

    return out


def safe_divide(a, b, eps=1e-6):
    return a / (b + eps)


def clip_pred(x):
    return np.clip(x, CFG.CLIP_LOW, CFG.CLIP_HIGH)


seed_everything(CFG.SEED)

In [5]:
# ============================================================
# 013: risk feature drop utility
# ============================================================

def drop_013_risk_features(feature_cols):
    """
    013 is a strict ablation from 012.
    Drop only IsOriginalData.
    Do not drop Race_Year / TyreAgeRatio_str / Race_Compound_Stint here.
    Those are separate ablations.
    """
    drop_set = set(getattr(CFG, "DROP_RISK_FEATURES", []))
    return [c for c in feature_cols if c not in drop_set]


def assert_013_feature_policy(feature_cols):
    """
    Guardrail for 013.
    This experiment is invalid if IsOriginalData remains in model features.
    """
    assert "IsOriginalData" not in feature_cols, (
        "013 policy violation: IsOriginalData is still in feature_cols."
    )

In [6]:
# ============================================================
# Load data
# ============================================================

print_section("Load Data")

COMP_PATH = first_existing_dir(CFG.COMP_PATHS)

train_raw = pd.read_csv(COMP_PATH / "train.csv")
test_raw = pd.read_csv(COMP_PATH / "test.csv")
sample_submission = pd.read_csv(COMP_PATH / "sample_submission.csv")

target_col_train = resolve_target_column(train_raw)
if target_col_train != CFG.TARGET:
    train_raw = train_raw.rename(columns={target_col_train: CFG.TARGET})

target_col_sub = resolve_target_column(sample_submission)
if target_col_sub != CFG.TARGET:
    sample_submission = sample_submission.rename(columns={target_col_sub: CFG.TARGET})

original_raw = None
original_path = first_existing_file(CFG.ORIGINAL_PATHS, required=False)

if CFG.USE_ORIGINAL and original_path is not None:
    original_raw = pd.read_csv(original_path)

    if CFG.TARGET not in original_raw.columns:
        original_target = resolve_target_column(original_raw)
        original_raw = original_raw.rename(columns={original_target: CFG.TARGET})

    print("Loaded original:", original_path, original_raw.shape)

print("COMP_PATH:", COMP_PATH)
print("train_raw:", train_raw.shape)
print("test_raw :", test_raw.shape)
print("sample_submission:", sample_submission.shape)
print("target mean train:", train_raw[CFG.TARGET].mean())

assert CFG.ID_COL in train_raw.columns
assert CFG.ID_COL in test_raw.columns
assert CFG.ID_COL in sample_submission.columns
assert CFG.TARGET in train_raw.columns
assert CFG.TARGET in sample_submission.columns
assert test_raw[CFG.ID_COL].equals(sample_submission[CFG.ID_COL])

if original_raw is not None:
    assert CFG.TARGET in original_raw.columns
    print("target mean original:", original_raw[CFG.TARGET].mean())

    if CFG.DANGER_COL in original_raw.columns:
        original_raw = original_raw.drop(columns=[CFG.DANGER_COL])
        print(f"Dropped danger column from original: {CFG.DANGER_COL}")

train_raw = reduce_mem_usage(train_raw)
test_raw = reduce_mem_usage(test_raw)

if original_raw is not None:
    original_raw = reduce_mem_usage(original_raw)

train_ids = train_raw[CFG.ID_COL].copy()
test_ids = test_raw[CFG.ID_COL].copy()


Load Data
Loaded original: /kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv (101371, 16)
COMP_PATH: /kaggle/input/competitions/playground-series-s6e5
train_raw: (439140, 16)
test_raw : (188165, 15)
sample_submission: (188165, 2)
target mean train: 0.19898210137996994
target mean original: 0.25479673673930414
Dropped danger column from original: Normalized_TyreLife


In [7]:
# ============================================================
# shared_004 style feature engineering
# ============================================================

BASE_CAT_COLS = ["Driver", "Compound", "Race"]

NUMERIC_BASE_COLS = [
    "Year",
    "PitStop",
    "LapNumber",
    "Stint",
    "TyreLife",
    "Position",
    "LapTime (s)",
    "LapTime_Delta",
    "Cumulative_Degradation",
    "RaceProgress",
    "Position_Change",
]


def add_domain_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    eps = 1e-6

    for col in BASE_CAT_COLS:
        if col in out.columns:
            out[col] = out[col].astype("string").fillna("__MISSING__").astype(str)

    if {"LapNumber", "RaceProgress"}.issubset(out.columns):
        out["EstimatedTotalLaps"] = safe_divide(
            out["LapNumber"],
            out["RaceProgress"].clip(lower=eps),
            eps,
        ).replace([np.inf, -np.inf], np.nan).clip(1, 120)

        out["LapsRemaining"] = (
            out["EstimatedTotalLaps"] - out["LapNumber"]
        ).clip(lower=0)

        out["LapProgress_x_LapNumber"] = out["LapNumber"] * out["RaceProgress"]

    if {"TyreLife", "LapNumber"}.issubset(out.columns):
        out["TyreAgeRatio"] = safe_divide(out["TyreLife"], out["LapNumber"].clip(lower=1), eps)
        out["LapPerTyreLife"] = safe_divide(out["LapNumber"], out["TyreLife"] + 1, eps)
        out["LapMinusTyreLife"] = out["LapNumber"] - out["TyreLife"]
        out["TyreLifeMinusLap"] = out["TyreLife"] - out["LapNumber"]

    if {"TyreLife", "EstimatedTotalLaps"}.issubset(out.columns):
        out["TyreAgeVsRace"] = safe_divide(
            out["TyreLife"],
            out["EstimatedTotalLaps"].clip(lower=1),
            eps,
        )

    if {"TyreLife", "RaceProgress"}.issubset(out.columns):
        out["PitWindowPressure"] = out["TyreLife"] * out["RaceProgress"]
        out["TyreLife_x_RaceProgress"] = out["TyreLife"] * out["RaceProgress"]

    if {"TyreLife", "LapsRemaining"}.issubset(out.columns):
        out["TyreLife_to_LapsRemaining"] = safe_divide(out["TyreLife"], out["LapsRemaining"] + 1, eps)
        out["LapsRemaining_to_TyreLife"] = safe_divide(out["LapsRemaining"], out["TyreLife"] + 1, eps)

    if {"Cumulative_Degradation", "TyreLife"}.issubset(out.columns):
        out["DegPerTyreLap"] = safe_divide(
            out["Cumulative_Degradation"],
            out["TyreLife"].clip(lower=1),
            eps,
        )
        out["AbsDegPerTyreLap"] = safe_divide(
            out["Cumulative_Degradation"].abs(),
            out["TyreLife"].clip(lower=1),
            eps,
        )

    if {"Cumulative_Degradation", "LapNumber"}.issubset(out.columns):
        out["DegPerRaceLap"] = safe_divide(
            out["Cumulative_Degradation"],
            out["LapNumber"].clip(lower=1),
            eps,
        )

    if {"LapTime_Delta", "TyreLife"}.issubset(out.columns):
        out["DeltaPerTyreLap"] = safe_divide(
            out["LapTime_Delta"],
            out["TyreLife"].clip(lower=1),
            eps,
        )
        out["AbsDeltaPerTyreLap"] = safe_divide(
            out["LapTime_Delta"].abs(),
            out["TyreLife"].clip(lower=1),
            eps,
        )

    if "LapTime_Delta" in out.columns:
        out["DeltaAbs"] = out["LapTime_Delta"].abs()
        out["LapTimeDeltaPositive"] = (out["LapTime_Delta"] > 0).astype(np.int8)
        out["LapTimeDeltaNegative"] = (out["LapTime_Delta"] < 0).astype(np.int8)

    if "Cumulative_Degradation" in out.columns:
        out["Abs_Cumulative_Degradation"] = out["Cumulative_Degradation"].abs()
        out["Positive_Degradation"] = (out["Cumulative_Degradation"] > 0).astype(np.int8)

    if "Position_Change" in out.columns:
        out["Abs_Position_Change"] = out["Position_Change"].abs()
        out["Gained_Position"] = (out["Position_Change"] > 0).astype(np.int8)
        out["Lost_Position"] = (out["Position_Change"] < 0).astype(np.int8)

    if {"Position", "RaceProgress"}.issubset(out.columns):
        out["PositionPressure"] = out["Position"] * out["RaceProgress"]

    if {"Stint", "TyreLife"}.issubset(out.columns):
        out["StintPressure"] = out["Stint"] * out["TyreLife"]
        out["TyreLife_x_Stint"] = out["TyreLife"] * out["Stint"]

    if {"Stint", "LapNumber"}.issubset(out.columns):
        out["Stint_x_LapNumber"] = out["Stint"] * out["LapNumber"]
        out["Is_First_Stint"] = (out["Stint"] == 1).astype(np.int8)
        out["Is_Late_Stint"] = (out["Stint"] >= 3).astype(np.int8)

    if "RaceProgress" in out.columns:
        out["Early_Race"] = (out["RaceProgress"] <= 0.25).astype(np.int8)
        out["Mid_Race"] = (
            (out["RaceProgress"] > 0.25)
            & (out["RaceProgress"] <= 0.65)
        ).astype(np.int8)
        out["Late_Race"] = (out["RaceProgress"] > 0.65).astype(np.int8)

        out["RacePhase"] = pd.cut(
            out["RaceProgress"],
            bins=[-np.inf, 0.20, 0.40, 0.60, 0.80, np.inf],
            labels=["P1", "P2", "P3", "P4", "P5"],
        ).astype(str)

    if "LapNumber" in out.columns:
        out["LapBin"] = pd.cut(
            out["LapNumber"],
            bins=[-np.inf, 5, 10, 20, 35, 50, np.inf],
            labels=[
                "L_000_005",
                "L_006_010",
                "L_011_020",
                "L_021_035",
                "L_036_050",
                "L_051_plus",
            ],
        ).astype(str)

    if "TyreLife" in out.columns:
        out["TyreLifeBin"] = pd.cut(
            out["TyreLife"],
            bins=[-np.inf, 3, 7, 12, 20, 30, np.inf],
            labels=[
                "T_000_003",
                "T_004_007",
                "T_008_012",
                "T_013_020",
                "T_021_030",
                "T_031_plus",
            ],
        ).astype(str)

    if "Position" in out.columns:
        out["PositionBin"] = pd.cut(
            out["Position"],
            bins=[-np.inf, 3, 8, 14, np.inf],
            labels=["front", "upper_mid", "lower_mid", "back"],
        ).astype(str)

    def make_cross(name, cols):
        if set(cols).issubset(out.columns):
            val = out[cols[0]].astype(str)
            for c in cols[1:]:
                val = val + "_" + out[c].astype(str)
            out[name] = val

    make_cross("Race_Year", ["Race", "Year"])
    make_cross("Compound_Stint", ["Compound", "Stint"])
    make_cross("Driver_Race", ["Driver", "Race"])
    make_cross("Driver_Compound", ["Driver", "Compound"])
    make_cross("Race_Compound", ["Race", "Compound"])
    make_cross("Race_Compound_Stint", ["Race", "Compound", "Stint"])
    make_cross("Compound_RacePhase", ["Compound", "RacePhase"])
    make_cross("Compound_TyreLifeBin", ["Compound", "TyreLifeBin"])
    make_cross("RacePhase_TyreLifeBin", ["RacePhase", "TyreLifeBin"])

    out = out.replace([np.inf, -np.inf], np.nan)

    for col in out.select_dtypes(include=["float64"]).columns:
        out[col] = out[col].astype(np.float32)

    return out


def add_digit_features(df: pd.DataFrame, numeric_cols=None, int_digit_limit=3, decimal_digit_limit=2) -> pd.DataFrame:
    out = df.copy()

    if numeric_cols is None:
        numeric_cols = out.select_dtypes(include=[np.number]).columns.tolist()

    for col in numeric_cols:
        if col not in out.columns:
            continue

        s = out[col].fillna(0).astype(float)
        abs_s = s.abs()

        for i in range(int_digit_limit):
            new_col = f"{col}_int_digit_{i + 1}"
            out[new_col] = ((abs_s // (10 ** i)) % 10).astype(np.int8)

        if pd.api.types.is_float_dtype(out[col]):
            for i in range(1, decimal_digit_limit + 1):
                new_col = f"{col}_dec_digit_{i}"
                out[new_col] = ((abs_s * (10 ** i)).round().astype(int) % 10).astype(np.int8)

    return out


def add_float_signature_features(df: pd.DataFrame, float_cols=None) -> pd.DataFrame:
    out = df.copy()

    selected = [
        "RaceProgress",
        "LapTime (s)",
        "LapTime_Delta",
        "Cumulative_Degradation",
        "TyreAgeRatio",
        "DegPerTyreLap",
        "DegPerRaceLap",
        "DeltaPerTyreLap",
        "DeltaAbs",
        "PitWindowPressure",
        "EstimatedTotalLaps",
        "LapsRemaining",
        "LapMinusTyreLife",
    ]

    float_cols = [c for c in selected if c in out.columns]

    for col in float_cols:
        scaled = (out[col].fillna(0).astype(float) * 100).round().astype(int).abs()

        for i in range(5):
            new_col = f"{col}_sig_{i + 1}"
            digit = ((scaled // (10 ** i)) % 10).astype(np.int8)

            if digit.nunique() > 1:
                out[new_col] = digit.astype(str)

    return out


def add_string_precision_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    if "RaceProgress" in out.columns:
        out["RaceProgress_str"] = out["RaceProgress"].round(4).astype(str)

    if "EstimatedTotalLaps" in out.columns:
        out["EstimatedTotalLaps_str"] = out["EstimatedTotalLaps"].round(1).astype(str)

    if "TyreAgeRatio" in out.columns:
        out["TyreAgeRatio_str"] = out["TyreAgeRatio"].round(3).astype(str)

    return out


def add_frequency_features(train_df, test_df, original_df=None):
    freq_cols = [
        "Driver",
        "Race",
        "Compound",
        "Race_Year",
        "Compound_Stint",
        "Driver_Race",
        "Driver_Compound",
        "Race_Compound",
        "Race_Compound_Stint",
        "Compound_RacePhase",
        "Compound_TyreLifeBin",
        "RacePhase_TyreLifeBin",
        "LapBin",
        "TyreLifeBin",
        "PositionBin",
    ]

    freq_cols = [c for c in freq_cols if c in train_df.columns and c in test_df.columns]

    frames = [train_df, test_df]
    if original_df is not None:
        frames.append(original_df)

    base = pd.concat([f[freq_cols] for f in frames], axis=0, ignore_index=True)
    total = len(base)

    for col in freq_cols:
        counts = base[col].astype(str).value_counts(dropna=False)

        for f in frames:
            f[f"{col}_count"] = f[col].astype(str).map(counts).fillna(0).astype(np.float32)
            f[f"{col}_freq"] = (f[f"{col}_count"] / total).astype(np.float32)

    del base
    gc.collect()

    if original_df is not None:
        return train_df, test_df, original_df

    return train_df, test_df, None


def add_light_group_stats(train_df, test_df, original_df=None):
    group_cols_list = [
        ["Race_Year"],
        ["Race_Compound_Stint"],
        ["Driver_Race"],
        ["Compound_Stint"],
    ]

    value_cols = [
        "LapTime_Delta",
        "Position_Change",
        "RaceProgress",
        "TyreLife",
    ]

    frames = [train_df, test_df]
    if original_df is not None:
        frames.append(original_df)

    for group_cols in group_cols_list:
        group_cols = [c for c in group_cols if all(c in f.columns for f in frames)]

        if not group_cols:
            continue

        group_name = "_".join(group_cols)

        for value_col in value_cols:
            if not all(value_col in f.columns for f in frames):
                continue

            base = pd.concat(
                [f[group_cols + [value_col]] for f in frames],
                axis=0,
                ignore_index=True,
            )

            stats = (
                base.groupby(group_cols, dropna=False)[value_col]
                .agg(["mean", "std"])
                .reset_index()
            )

            stats.columns = group_cols + [
                f"{value_col}_mean_by_{group_name}",
                f"{value_col}_std_by_{group_name}",
            ]

            for idx, f in enumerate(frames):
                frames[idx] = f.merge(stats, on=group_cols, how="left")

                mean_col = f"{value_col}_mean_by_{group_name}"
                frames[idx][f"{value_col}_diff_mean_by_{group_name}"] = (
                    frames[idx][value_col] - frames[idx][mean_col]
                ).astype(np.float32)

            del base, stats
            gc.collect()

    if original_df is not None:
        return frames[0], frames[1], frames[2]

    return frames[0], frames[1], None


def clean_columns(train_df, test_df, original_df=None):
    train_df = train_df.copy()
    test_df = test_df.copy()

    if original_df is not None:
        original_df = original_df.copy()

    common_cols = [c for c in train_df.columns if c in test_df.columns]

    train_df = train_df[common_cols + [CFG.TARGET]]
    test_df = test_df[common_cols]

    if original_df is not None:
        for c in common_cols:
            if c not in original_df.columns:
                original_df[c] = np.nan

        original_df = original_df[common_cols + [CFG.TARGET]]

    return train_df, test_df, original_df


def fill_missing_and_types(train_df, test_df, original_df=None):
    frames = [train_df, test_df]
    if original_df is not None:
        frames.append(original_df)

    all_feature_cols = [c for c in train_df.columns if c != CFG.TARGET]

    cat_cols = []

    for col in all_feature_cols:
        if any(
            (
                f[col].dtype == "object"
                or str(f[col].dtype).startswith("category")
                or str(f[col].dtype).startswith("string")
            )
            for f in frames
            if col in f.columns
        ):
            cat_cols.append(col)

    num_cols = [c for c in all_feature_cols if c not in cat_cols]

    for col in cat_cols:
        values = pd.concat(
            [f[col].astype("string") for f in frames if col in f.columns],
            axis=0,
        )
        mode_values = values.mode()
        mode_value = mode_values.iloc[0] if len(mode_values) else "__MISSING__"

        for f in frames:
            if col in f.columns:
                f[col] = f[col].astype("string").fillna(mode_value).astype(str)

    for col in num_cols:
        values = pd.concat([f[col] for f in frames if col in f.columns], axis=0)
        fill_value = values.median()

        for f in frames:
            if col in f.columns:
                f[col] = f[col].replace([np.inf, -np.inf], np.nan).fillna(fill_value)
                if f[col].dtype == "float64":
                    f[col] = f[col].astype(np.float32)

    train_df = reduce_mem_usage(train_df)
    test_df = reduce_mem_usage(test_df)

    if original_df is not None:
        original_df = reduce_mem_usage(original_df)

    return train_df, test_df, original_df, cat_cols

In [8]:
# ============================================================
# Feature Engineering
# ============================================================

print_section("Feature Engineering")

train_fe = train_raw.copy()
test_fe = test_raw.copy()

if original_raw is not None and CFG.TARGET in original_raw.columns:
    original_fe = original_raw.copy()
else:
    original_fe = None

if original_fe is not None and CFG.DANGER_COL in original_fe.columns:
    original_fe = original_fe.drop(columns=[CFG.DANGER_COL])
    print(f"Dropped danger column from original_fe: {CFG.DANGER_COL}")

train_fe["IsOriginalData"] = 0
test_fe["IsOriginalData"] = 0

if original_fe is not None:
    original_fe["IsOriginalData"] = 1

train_fe = add_domain_features(train_fe)
test_fe = add_domain_features(test_fe)

if original_fe is not None:
    original_fe = add_domain_features(original_fe)

digit_source_cols = [
    "Year",
    "PitStop",
    "LapNumber",
    "Stint",
    "TyreLife",
    "Position",
    "LapTime (s)",
    "LapTime_Delta",
    "Cumulative_Degradation",
    "RaceProgress",
    "Position_Change",
    "EstimatedTotalLaps",
    "LapsRemaining",
    "TyreAgeRatio",
    "DegPerTyreLap",
    "DegPerRaceLap",
    "DeltaPerTyreLap",
    "DeltaAbs",
    "PositionPressure",
    "StintPressure",
    "PitWindowPressure",
    "LapMinusTyreLife",
]
digit_source_cols = [c for c in digit_source_cols if c in train_fe.columns and c in test_fe.columns]

train_fe = add_digit_features(train_fe, digit_source_cols)
test_fe = add_digit_features(test_fe, digit_source_cols)

if original_fe is not None:
    original_fe = add_digit_features(original_fe, digit_source_cols)

train_fe = add_float_signature_features(train_fe)
test_fe = add_float_signature_features(test_fe)

if original_fe is not None:
    original_fe = add_float_signature_features(original_fe)

train_fe = add_string_precision_features(train_fe)
test_fe = add_string_precision_features(test_fe)

if original_fe is not None:
    original_fe = add_string_precision_features(original_fe)

train_fe, test_fe, original_fe = add_frequency_features(train_fe, test_fe, original_fe)
train_fe, test_fe, original_fe = add_light_group_stats(train_fe, test_fe, original_fe)

train_fe, test_fe, original_fe = clean_columns(train_fe, test_fe, original_fe)
train_fe, test_fe, original_fe, cat_cols = fill_missing_and_types(train_fe, test_fe, original_fe)

print("train_fe:", train_fe.shape)
print("test_fe :", test_fe.shape)

if original_fe is not None:
    print("original_fe:", original_fe.shape)

print("categorical features:", len(cat_cols))


Feature Engineering
train_fe: (439140, 303)
test_fe : (188165, 302)
original_fe: (101371, 303)
categorical features: 77


In [9]:
# ============================================================
# Prepare matrices
# ============================================================

print_section("Prepare Matrices")

X_comp = train_fe.drop(columns=[CFG.TARGET, CFG.ID_COL], errors="ignore")
y_comp = train_fe[CFG.TARGET].astype(int).reset_index(drop=True)

X_test_final = test_fe.drop(columns=[CFG.ID_COL], errors="ignore")

if original_fe is not None:
    X_orig = original_fe.drop(columns=[CFG.TARGET, CFG.ID_COL], errors="ignore")
    y_orig = original_fe[CFG.TARGET].astype(int).reset_index(drop=True)
else:
    X_orig = None
    y_orig = None

common_features = [c for c in X_comp.columns if c in X_test_final.columns]

# 013差分: IsOriginalDataだけを除外
common_features = drop_013_risk_features(common_features)
assert_013_feature_policy(common_features)

print("013 dropped risk features:", CFG.DROP_RISK_FEATURES)
print("final feature count after 013 drop:", len(common_features))
print("IsOriginalData in features:", "IsOriginalData" in common_features)

X_comp = X_comp[common_features].reset_index(drop=True)
X_test_final = X_test_final[common_features].reset_index(drop=True)

if X_orig is not None:
    for c in common_features:
        if c not in X_orig.columns:
            X_orig[c] = np.nan
    X_orig = X_orig[common_features].reset_index(drop=True)

cat_cols = [c for c in cat_cols if c in common_features]
cat_cols = [c for c in cat_cols if c not in CFG.DROP_RISK_FEATURES]
cat_features = [X_comp.columns.get_loc(c) for c in cat_cols]

print("X_comp:", X_comp.shape)
print("X_test_final:", X_test_final.shape)

if X_orig is not None:
    print("X_orig:", X_orig.shape)
    print("original target mean:", float(y_orig.mean()))

print("target mean competition:", float(y_comp.mean()))
print("n_cat_features:", len(cat_features))
print("n_features:", len(common_features))


Prepare Matrices
013 dropped risk features: ['IsOriginalData']
final feature count after 013 drop: 300
IsOriginalData in features: False
X_comp: (439140, 300)
X_test_final: (188165, 300)
X_orig: (101371, 300)
original target mean: 0.25479673673930414
target mean competition: 0.19898210137996994
n_cat_features: 77
n_features: 300


In [10]:
# ============================================================
# CatBoost model
# ============================================================

def get_catboost_params(seed: int, iterations: int, task_type: str):
    params = {
        "iterations": iterations,
        "learning_rate": CFG.LEARNING_RATE,
        "depth": CFG.DEPTH,
        "l2_leaf_reg": CFG.L2_LEAF_REG,
        "random_strength": CFG.RANDOM_STRENGTH,

        # Fixed to avoid CatBoost CPU/GPU hidden-default mismatch.
        "bootstrap_type": CFG.BOOTSTRAP_TYPE,
        "subsample": CFG.SUBSAMPLE,
        "border_count": CFG.BORDER_COUNT,

        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "auto_class_weights": CFG.AUTO_CLASS_WEIGHTS,

        "task_type": task_type,
        "random_seed": seed,
        "early_stopping_rounds": CFG.EARLY_STOPPING_ROUNDS,
        "allow_writing_files": False,
        "verbose": CFG.VERBOSE,
    }

    if task_type == "GPU":
        params["devices"] = CFG.DEVICES
    else:
        params["thread_count"] = -1

    return params


def fit_catboost_with_fallback(X_tr, y_tr, X_va, y_va, seed):
    params = get_catboost_params(seed=seed, iterations=CFG.ITERATIONS, task_type=CFG.TASK_TYPE)

    model = CatBoostClassifier(**params)

    try:
        model.fit(
            X_tr,
            y_tr,
            eval_set=(X_va, y_va),
            cat_features=cat_features,
            use_best_model=True,
        )
        used_device = CFG.TASK_TYPE
        return model, used_device

    except Exception as e:
        print("CatBoost GPU/device training failed. Retrying with CPU.")
        print("Error:", repr(e))

        cpu_params = get_catboost_params(seed=seed, iterations=CFG.ITERATIONS, task_type="CPU")
        cpu_model = CatBoostClassifier(**cpu_params)

        cpu_model.fit(
            X_tr,
            y_tr,
            eval_set=(X_va, y_va),
            cat_features=cat_features,
            use_best_model=True,
        )

        return cpu_model, "CPU"

In [11]:
# ============================================================
# 5fold OOF training
# ============================================================

print_section("5fold OOF Training")

folds = list(
    StratifiedKFold(
        n_splits=CFG.N_SPLITS,
        shuffle=True,
        random_state=CFG.SEED,
    ).split(X_comp, y_comp)
)

oof = np.zeros(len(X_comp), dtype=np.float32)
pred = np.zeros(len(X_test_final), dtype=np.float32)

fold_records = []
feature_importance_records = []
used_devices = []

for fold, (tr_idx, va_idx) in enumerate(folds, 1):
    print("\n" + "=" * 90)
    print(f"Fold {fold}/{CFG.N_SPLITS}")
    print("=" * 90)

    X_tr_comp = X_comp.iloc[tr_idx].copy().reset_index(drop=True)
    y_tr_comp = y_comp.iloc[tr_idx].copy().reset_index(drop=True)

    X_va = X_comp.iloc[va_idx].copy().reset_index(drop=True)
    y_va = y_comp.iloc[va_idx].copy().reset_index(drop=True)

    if X_orig is not None and CFG.USE_ORIGINAL:
        X_tr = pd.concat(
            [X_tr_comp, X_orig],
            axis=0,
            ignore_index=True,
        )
        y_tr = pd.concat(
            [y_tr_comp, y_orig],
            axis=0,
            ignore_index=True,
        )
    else:
        X_tr = X_tr_comp
        y_tr = y_tr_comp

    print("X_tr:", X_tr.shape, "target mean:", float(y_tr.mean()))
    print("X_va:", X_va.shape, "target mean:", float(y_va.mean()))
    print("X_test:", X_test_final.shape)

    model, used_device = fit_catboost_with_fallback(
        X_tr=X_tr,
        y_tr=y_tr,
        X_va=X_va,
        y_va=y_va,
        seed=CFG.SEED + fold,
    )

    va_pred = clip_pred(model.predict_proba(X_va)[:, 1]).astype(np.float32)
    te_pred = clip_pred(model.predict_proba(X_test_final)[:, 1]).astype(np.float32)

    oof[va_idx] = va_pred
    pred += te_pred / CFG.N_SPLITS

    fold_auc = roc_auc_score(y_va, va_pred)
    fold_logloss = log_loss(y_va, va_pred)
    best_iter = int(model.get_best_iteration() or -1)

    print(f"Fold {fold} AUC    : {fold_auc:.9f}")
    print(f"Fold {fold} LogLoss: {fold_logloss:.9f}")
    print(f"Fold {fold} best_iter: {best_iter}")
    print(f"Fold {fold} used_device: {used_device}")

    fold_records.append({
        "fold": fold,
        "auc": float(fold_auc),
        "logloss": float(fold_logloss),
        "best_iteration": best_iter,
        "used_device": used_device,
        "n_train_comp": int(len(X_tr_comp)),
        "n_train_orig": int(len(X_orig)) if X_orig is not None and CFG.USE_ORIGINAL else 0,
        "n_valid": int(len(X_va)),
        "n_features": int(X_tr.shape[1]),
        "n_cat_features": int(len(cat_features)),
    })

    fi = pd.DataFrame({
        "feature": X_tr.columns,
        "importance": model.get_feature_importance(),
        "fold": fold,
    })
    feature_importance_records.append(fi)

    used_devices.append(used_device)

    del model, X_tr_comp, y_tr_comp, X_va, y_va, X_tr, y_tr, va_pred, te_pred
    gc.collect()


5fold OOF Training

Fold 1/5
X_tr: (452683, 300) target mean: 0.21147911452385001
X_va: (87828, 300) target mean: 0.19899121009245344
X_test: (188165, 300)


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9244625	best: 0.9244625 (0)	total: 426ms	remaining: 1h 3m 51s
200:	test: 0.9479572	best: 0.9479572 (200)	total: 43.5s	remaining: 31m 45s
400:	test: 0.9503868	best: 0.9503868 (400)	total: 1m 27s	remaining: 31m 13s
600:	test: 0.9514022	best: 0.9514022 (600)	total: 2m 13s	remaining: 31m 9s
800:	test: 0.9518824	best: 0.9518824 (800)	total: 3m	remaining: 30m 43s
1000:	test: 0.9522270	best: 0.9522270 (1000)	total: 3m 46s	remaining: 30m 10s
1200:	test: 0.9524792	best: 0.9524792 (1200)	total: 4m 33s	remaining: 29m 35s
1400:	test: 0.9526916	best: 0.9526953 (1398)	total: 5m 19s	remaining: 28m 54s
1600:	test: 0.9527932	best: 0.9527932 (1600)	total: 6m 6s	remaining: 28m 13s
1800:	test: 0.9528914	best: 0.9528940 (1794)	total: 6m 52s	remaining: 27m 30s
2000:	test: 0.9529638	best: 0.9529638 (2000)	total: 7m 38s	remaining: 26m 44s
2200:	test: 0.9530757	best: 0.9530757 (2200)	total: 8m 25s	remaining: 26m 2s
2400:	test: 0.9531441	best: 0.9531441 (2400)	total: 9m 11s	remaining: 25m 16s
2600:	t

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9215398	best: 0.9215398 (0)	total: 307ms	remaining: 46m 4s
200:	test: 0.9457499	best: 0.9457499 (200)	total: 44.7s	remaining: 32m 35s
400:	test: 0.9483535	best: 0.9483535 (400)	total: 1m 28s	remaining: 31m 39s
600:	test: 0.9494182	best: 0.9494182 (600)	total: 2m 15s	remaining: 31m 33s
800:	test: 0.9499457	best: 0.9499457 (800)	total: 3m 1s	remaining: 31m
1000:	test: 0.9502990	best: 0.9503006 (999)	total: 3m 48s	remaining: 30m 28s
1200:	test: 0.9505362	best: 0.9505362 (1200)	total: 4m 35s	remaining: 29m 49s
1400:	test: 0.9507077	best: 0.9507077 (1400)	total: 5m 21s	remaining: 29m 1s
1600:	test: 0.9508721	best: 0.9508721 (1600)	total: 6m 7s	remaining: 28m 18s
1800:	test: 0.9510210	best: 0.9510249 (1796)	total: 6m 54s	remaining: 27m 36s
2000:	test: 0.9511108	best: 0.9511126 (1992)	total: 7m 40s	remaining: 26m 50s
2200:	test: 0.9511799	best: 0.9511799 (2200)	total: 8m 27s	remaining: 26m 8s
2400:	test: 0.9512244	best: 0.9512258 (2397)	total: 9m 13s	remaining: 25m 22s
2600:	test: 

...

0:	test: 0.9279130	best: 0.9279130 (0)	total: 317ms	remaining: 47m 36s
200:	test: 0.9472563	best: 0.9472603 (199)	total: 44.5s	remaining: 32m 29s
400:	test: 0.9497746	best: 0.9497746 (400)	total: 1m 28s	remaining: 31m 37s
600:	test: 0.9507364	best: 0.9507364 (600)	total: 2m 14s	remaining: 31m 24s
800:	test: 0.9512416	best: 0.9512419 (797)	total: 3m 1s	remaining: 30m 54s
1000:	test: 0.9515432	best: 0.9515437 (999)	total: 3m 47s	remaining: 30m 19s
1200:	test: 0.9517809	best: 0.9517809 (1199)	total: 4m 34s	remaining: 29m 41s
1400:	test: 0.9519230	best: 0.9519230 (1400)	total: 5m 20s	remaining: 28m 56s
1600:	test: 0.9520673	best: 0.9520673 (1600)	total: 6m 6s	remaining: 28m 14s
1800:	test: 0.9521518	best: 0.9521571 (1784)	total: 6m 53s	remaining: 27m 33s
2000:	test: 0.9522329	best: 0.9522360 (1997)	total: 7m 40s	remaining: 26m 51s
2200:	test: 0.9522728	best: 0.9522752 (2197)	total: 8m 27s	remaining: 26m 8s
2400:	test: 0.9523128	best: 0.9523129 (2399)	total: 9m 14s	remaining: 25m 24s
2600:	

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9246763	best: 0.9246763 (0)	total: 297ms	remaining: 44m 28s
200:	test: 0.9457906	best: 0.9457906 (200)	total: 44.1s	remaining: 32m 9s
400:	test: 0.9482555	best: 0.9482555 (400)	total: 1m 28s	remaining: 31m 31s
600:	test: 0.9493813	best: 0.9493813 (600)	total: 2m 15s	remaining: 31m 31s
800:	test: 0.9500519	best: 0.9500519 (800)	total: 3m 1s	remaining: 31m 1s
1000:	test: 0.9503400	best: 0.9503400 (1000)	total: 3m 48s	remaining: 30m 23s
1200:	test: 0.9506207	best: 0.9506207 (1200)	total: 4m 34s	remaining: 29m 40s
1400:	test: 0.9508278	best: 0.9508278 (1400)	total: 5m 19s	remaining: 28m 54s
1600:	test: 0.9509623	best: 0.9509623 (1600)	total: 6m 6s	remaining: 28m 13s
1800:	test: 0.9510828	best: 0.9510828 (1800)	total: 6m 53s	remaining: 27m 32s
2000:	test: 0.9511728	best: 0.9511772 (1994)	total: 7m 39s	remaining: 26m 47s
2200:	test: 0.9512627	best: 0.9512627 (2200)	total: 8m 26s	remaining: 26m 5s
2400:	test: 0.9513237	best: 0.9513237 (2400)	total: 9m 13s	remaining: 25m 20s
2600:	t

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9235175	best: 0.9235175 (0)	total: 297ms	remaining: 44m 30s
200:	test: 0.9474851	best: 0.9474851 (200)	total: 45.2s	remaining: 32m 58s
400:	test: 0.9500231	best: 0.9500231 (400)	total: 1m 29s	remaining: 31m 53s
600:	test: 0.9510520	best: 0.9510520 (600)	total: 2m 16s	remaining: 31m 46s
800:	test: 0.9515528	best: 0.9515554 (798)	total: 3m 2s	remaining: 31m 6s
1000:	test: 0.9519288	best: 0.9519288 (1000)	total: 3m 48s	remaining: 30m 28s
1200:	test: 0.9521769	best: 0.9521769 (1200)	total: 4m 34s	remaining: 29m 45s
1400:	test: 0.9523857	best: 0.9523857 (1400)	total: 5m 21s	remaining: 29m 3s
1600:	test: 0.9525105	best: 0.9525113 (1599)	total: 6m 8s	remaining: 28m 21s
1800:	test: 0.9525974	best: 0.9526001 (1778)	total: 6m 54s	remaining: 27m 35s
2000:	test: 0.9527098	best: 0.9527098 (2000)	total: 7m 41s	remaining: 26m 54s
2200:	test: 0.9527784	best: 0.9527784 (2200)	total: 8m 28s	remaining: 26m 11s
2400:	test: 0.9528467	best: 0.9528467 (2400)	total: 9m 15s	remaining: 25m 27s
2600:	

In [12]:
# ============================================================
# CV result
# ============================================================

print_section("CV Result")

cv_auc = roc_auc_score(y_comp, oof)
cv_logloss = log_loss(y_comp, clip_pred(oof))

fold_df = pd.DataFrame(fold_records)

print(f"Overall CV AUC    : {cv_auc:.9f}")
print(f"Overall CV LogLoss: {cv_logloss:.9f}")
print("fold auc mean:", fold_df["auc"].mean())
print("fold auc std :", fold_df["auc"].std())
print("used devices :", sorted(set(used_devices)))

print(fold_df)


CV Result
Overall CV AUC    : 0.952347506
Overall CV LogLoss: 0.261310733
fold auc mean: 0.9523726091553495
fold auc std : 0.0008497462887084124
used devices : ['GPU']
   fold       auc   logloss  best_iteration used_device  n_train_comp  \
0     1  0.953337  0.258612            5043         GPU        351312   
1     2  0.951417  0.264663            3637         GPU        351312   
2     3  0.952454  0.264805            3528         GPU        351312   
3     4  0.951609  0.259497            5399         GPU        351312   
4     5  0.953046  0.258977            3713         GPU        351312   

   n_train_orig  n_valid  n_features  n_cat_features  
0        101371    87828         300              77  
1        101371    87828         300              77  
2        101371    87828         300              77  
3        101371    87828         300              77  
4        101371    87828         300              77  


In [13]:
# ============================================================
# Save artifacts
# ============================================================

print_section("Save Artifacts")

np.save(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.npy", oof)
np.save(CFG.OUTDIR / f"pred_{CFG.EXP_ID}.npy", pred)

oof_csv = pd.DataFrame({
    CFG.ID_COL: train_ids,
    CFG.TARGET: y_comp.values,
    "pred": oof,
})
oof_csv.to_csv(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.csv", index=False)

sub = sample_submission.copy()
sub[CFG.TARGET] = clip_pred(pred)

sub_path = CFG.OUTDIR / f"submission_{CFG.EXP_ID}.csv"
sub.to_csv(sub_path, index=False)

fold_df.to_csv(CFG.OUTDIR / "fold_scores.csv", index=False)

fi_all = pd.concat(feature_importance_records, axis=0, ignore_index=True)
fi_summary = (
    fi_all
    .groupby("feature", as_index=False)["importance"]
    .agg(["mean", "std"])
    .reset_index()
    .sort_values("mean", ascending=False)
)
fi_all.to_csv(CFG.OUTDIR / "feature_importance_by_fold.csv", index=False)
fi_summary.to_csv(CFG.OUTDIR / "feature_importance.csv", index=False)

feature_df = pd.DataFrame({
    "feature": common_features,
    "is_cat": [c in cat_cols for c in common_features],
    "is_count": [c.endswith("_count") for c in common_features],
    "is_freq": [c.endswith("_freq") for c in common_features],
    "is_digit": [
        ("_int_digit_" in c) or ("_dec_digit_" in c) or ("_sig_" in c)
        for c in common_features
    ],
    "is_string_precision": [
        c in ["RaceProgress_str", "EstimatedTotalLaps_str", "TyreAgeRatio_str"]
        for c in common_features
    ],
    "is_group_stat": [
        ("_mean_by_" in c) or ("_std_by_" in c) or ("_diff_mean_by_" in c)
        for c in common_features
    ],
    "contains_year": ["Year" in c for c in common_features],
    "contains_race_year": ["Race_Year" in c for c in common_features],
})
feature_df.to_csv(CFG.OUTDIR / "feature_columns.csv", index=False)

cat_feature_df = pd.DataFrame({
    "cat_feature": cat_cols,
    "index": cat_features,
})
cat_feature_df.to_csv(CFG.OUTDIR / "categorical_features.csv", index=False)

result = {
    "experiment": {
        "id": CFG.EXP_ID,
        "competition": CFG.COMPETITION,
        "metric": CFG.METRIC,
    },
    "source": {
        "shared_code": "shared_004",
        "conversion": "single holdout CatBoost code converted to 5fold OOF/pred artifact generator",
        "note": [
            "This is not shared_004 as-is submission.",
            "The implementation keeps shared_004 large-FE design and converts it to OOF artifacts.",
            "Original rows are appended to each fold train side only.",
            "Competition validation rows only are used for OOF.",
        ],
    },
    "data": {
        "train_shape": list(train_raw.shape),
        "test_shape": list(test_raw.shape),
        "original_available": original_raw is not None,
        "original_shape_after_drop": list(original_raw.shape) if original_raw is not None else None,
        "target": CFG.TARGET,
        "id_col": CFG.ID_COL,
        "danger_col": CFG.DANGER_COL,
        "danger_col_used": False,
        "use_original_rows": CFG.USE_ORIGINAL,
        "competition_target_mean": float(train_raw[CFG.TARGET].mean()),
        "original_target_mean": float(original_raw[CFG.TARGET].mean()) if original_raw is not None else None,
    },
    "features": {
        "n_features": int(len(common_features)),
        "n_cat_features": int(len(cat_cols)),
        "feature_blocks": [
            "domain_features",
            "digit_features",
            "float_signature_features",
            "string_precision_features",
            "frequency_features",
            "light_group_stats",
        ],
        "important_risk_features_to_watch": [
            "Race_Year",
            "IsOriginalData",
            "TyreAgeRatio_str",
            "Race_Compound_Stint",
            "Race_Year group stats",
        ],
        "normalized_tyrelife_policy": [
            "Normalized_TyreLife is dropped.",
            "Direct use is forbidden.",
            "Intentional proxy reconstruction is not added.",
        ],
        "dropped_risk_features_013": CFG.DROP_RISK_FEATURES,
        "isoriginaldata_used": False,
    },
    "model": {
        "family": "CatBoost",
        "estimator": "CatBoostClassifier",
        "params": {
            "iterations": CFG.ITERATIONS,
            "learning_rate": CFG.LEARNING_RATE,
            "depth": CFG.DEPTH,
            "l2_leaf_reg": CFG.L2_LEAF_REG,
            "random_strength": CFG.RANDOM_STRENGTH,
            "bootstrap_type": CFG.BOOTSTRAP_TYPE,
            "subsample": CFG.SUBSAMPLE,
            "border_count": CFG.BORDER_COUNT,
            "loss_function": "Logloss",
            "eval_metric": "AUC",
            "auto_class_weights": CFG.AUTO_CLASS_WEIGHTS,
            "early_stopping_rounds": CFG.EARLY_STOPPING_ROUNDS,
            "task_type_requested": CFG.TASK_TYPE,
            "devices_requested": CFG.DEVICES,
            "devices_used": sorted(set(used_devices)),
        },
        "cv": {
            "scheme": "StratifiedKFold",
            "n_splits": CFG.N_SPLITS,
            "shuffle": True,
            "random_state": CFG.SEED,
        },
    },
    "result": {
        "cv_auc": float(cv_auc),
        "cv_logloss": float(cv_logloss),
        "fold_auc_mean": float(fold_df["auc"].mean()),
        "fold_auc_std": float(fold_df["auc"].std()),
        "fold_scores": fold_records,
    },
    "artifacts": {
        "outdir": str(CFG.OUTDIR),
        "oof_npy": str(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.npy"),
        "pred_npy": str(CFG.OUTDIR / f"pred_{CFG.EXP_ID}.npy"),
        "oof_csv": str(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.csv"),
        "submission": str(sub_path),
        "fold_scores": str(CFG.OUTDIR / "fold_scores.csv"),
        "feature_columns": str(CFG.OUTDIR / "feature_columns.csv"),
        "categorical_features": str(CFG.OUTDIR / "categorical_features.csv"),
        "feature_importance": str(CFG.OUTDIR / "feature_importance.csv"),
        "cv_result": str(CFG.OUTDIR / "cv_result.json"),
    },
    "decision_policy": {
        "single_model": [
            "Compare against 006b, 007, 008, 009.",
            "If CV/LB are strong, submit once.",
            "Do not adopt only by single CV; test blend conversion.",
        ],
        "blend": [
            "Add to 010 core: 006b + 007 + 008 + 009 + 012.",
            "Evaluate avg / Ridge / ElasticNet / LogReg / HC / NM.",
            "If 012 receives natural positive weight and improves LB/CV, keep.",
            "If 012 is high CV but zero-weight in HC/NM and weak in LogReg, hold or drop.",
        ],
    },
}

with open(CFG.OUTDIR / "cv_result.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print("Saved to:", CFG.OUTDIR)
print("Submission:", sub_path)
print(json.dumps(result, ensure_ascii=False, indent=2))

print_section("Top Feature Importance")
display(fi_summary.head(80))

print_section("Submission Preview")
display(sub.head())


Save Artifacts
Saved to: /kaggle/working/exp_20260505_013_cat_shared004_no_isoriginaldata_min
Submission: /kaggle/working/exp_20260505_013_cat_shared004_no_isoriginaldata_min/submission_exp_20260505_013_cat_shared004_no_isoriginaldata_min.csv
{
  "experiment": {
    "id": "exp_20260505_013_cat_shared004_no_isoriginaldata_min",
    "competition": "PS S6E5 Predicting F1 Pit Stops",
    "metric": "AUC"
  },
  "source": {
    "shared_code": "shared_004",
    "conversion": "single holdout CatBoost code converted to 5fold OOF/pred artifact generator",
    "note": [
      "This is not shared_004 as-is submission.",
      "The implementation keeps shared_004 large-FE design and converts it to OOF artifacts.",
      "Original rows are appended to each fold train side only.",
      "Competition validation rows only are used for OOF."
    ]
  },
  "data": {
    "train_shape": [
      439140,
      16
    ],
    "test_shape": [
      188165,
      15
    ],
    "original_available": true,
    "or

,index,feature,mean,std
243,243,Race_Year,8.295749,0.640803
205,205,Position_Change_std_by_Race_Year,4.922266,1.647355
269,269,TyreAgeRatio_str,4.073639,0.363416
238,238,Race_Compound_Stint,3.601457,0.454827
49,49,DeltaAbs,2.189524,0.251873
...,...,...,...,...
199,199,Position_Change_mean_by_Driver_Race,0.423741,0.073054
204,204,Position_Change_std_by_Race_Compound_Stint,0.411120,0.040317
76,76,Driver_Race_count,0.407519,0.057500
258,258,Stint_x_LapNumber,0.405392,0.039665



Submission Preview


,id,PitNextLap
0,439140,0.020078
1,439141,0.046284
2,439142,0.011379
3,439143,0.468278
4,439144,0.945307
